# Advanced SQL

This notebook covers advanced SQL concepts: JOINs, Subqueries, CTEs, and Set Operations.

> **Note:** Since Jupyter Notebooks don't natively support SQL, we use the `%%sql` magic command. This requires loading the SQL extension first (`%load_ext sql`) and connecting to a DuckDB database. Each SQL cell must start with `%%sql` to be interpreted as SQL rather than Python.

## Setup

In [23]:
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False
%sql duckdb:///:memory:

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


## Sample Data Setup

Let's create related tables to demonstrate JOINs.

In [24]:
%%sql
CREATE TABLE customers (
    customer_id INTEGER,
    customer_name VARCHAR,
    city VARCHAR,
    country VARCHAR
);

CREATE TABLE orders (
    order_id INTEGER,
    customer_id INTEGER,
    order_date DATE,
    amount DECIMAL(10,2)
);

CREATE TABLE products (
    product_id INTEGER,
    product_name VARCHAR,
    category VARCHAR,
    price DECIMAL(10,2)
);

CREATE TABLE order_items (
    order_id INTEGER,
    product_id INTEGER,
    quantity INTEGER
);

-- Insert sample data
INSERT INTO customers VALUES 
    (1, 'Acme Corp', 'New York', 'USA'),
    (2, 'TechStart Ltd', 'London', 'UK'),
    (3, 'Global Industries', 'Tokyo', 'Japan'),
    (4, 'Innovate GmbH', 'Berlin', 'Germany'),
    (5, 'DataCo', 'San Francisco', 'USA');

INSERT INTO orders VALUES 
    (101, 1, '2024-01-15', 1500.00),
    (102, 1, '2024-02-20', 2300.00),
    (103, 2, '2024-01-25', 950.00),
    (104, 3, '2024-03-10', 1800.00),
    (105, 2, '2024-03-15', 1200.00);

INSERT INTO products VALUES 
    (1, 'Laptop Pro', 'Electronics', 1200.00),
    (2, 'Wireless Mouse', 'Electronics', 25.00),
    (3, 'Office Chair', 'Furniture', 350.00),
    (4, 'Standing Desk', 'Furniture', 600.00),
    (5, 'Monitor 27"', 'Electronics', 400.00);

INSERT INTO order_items VALUES 
    (101, 1, 1), (101, 2, 3),
    (102, 4, 2), (102, 5, 2),
    (103, 2, 5), (103, 3, 1),
    (104, 1, 1), (104, 5, 1),
    (105, 3, 2), (105, 4, 1);

,Success


## 1. JOIN Operations

JOINs combine rows from multiple tables based on related columns.

### 1.1 INNER JOIN

Returns rows that have matching values in both tables.

In [25]:
%%sql
-- Join customers with their orders
SELECT 
    c.customer_name,
    c.city,
    o.order_id,
    o.order_date,
    o.amount
FROM customers c
INNER JOIN orders o ON c.customer_id = o.customer_id
ORDER BY o.order_date;

,customer_name,city,order_id,order_date,amount
0,Acme Corp,New York,101,2024-01-15,1500.0
1,TechStart Ltd,London,103,2024-01-25,950.0
2,Acme Corp,New York,102,2024-02-20,2300.0
3,Global Industries,Tokyo,104,2024-03-10,1800.0
4,TechStart Ltd,London,105,2024-03-15,1200.0


### 1.2 LEFT JOIN

Returns all rows from the left table, and matching rows from the right table (NULL if no match).

In [26]:
%%sql
-- All customers, including those without orders
SELECT 
    c.customer_name,
    c.city,
    o.order_id,
    o.amount
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
ORDER BY c.customer_name;

,customer_name,city,order_id,amount
0,Acme Corp,New York,102,2300.0
1,Acme Corp,New York,101,1500.0
2,DataCo,San Francisco,<NA>,NaN
3,Global Industries,Tokyo,104,1800.0
4,Innovate GmbH,Berlin,<NA>,NaN
5,TechStart Ltd,London,105,1200.0
6,TechStart Ltd,London,103,950.0


### 1.3 Multiple JOINs

Join more than two tables together.

In [27]:
%%sql
-- Complete order details with customer and product information
SELECT 
    c.customer_name,
    o.order_id,
    o.order_date,
    p.product_name,
    p.category,
    oi.quantity,
    p.price,
    (oi.quantity * p.price) AS line_total
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
ORDER BY o.order_id, p.product_name;

,customer_name,order_id,order_date,product_name,category,quantity,price,line_total
0,Acme Corp,101,2024-01-15,Laptop Pro,Electronics,1,1200.0,1200.0
1,Acme Corp,101,2024-01-15,Wireless Mouse,Electronics,3,25.0,75.0
2,Acme Corp,102,2024-02-20,"Monitor 27""",Electronics,2,400.0,800.0
3,Acme Corp,102,2024-02-20,Standing Desk,Furniture,2,600.0,1200.0
4,TechStart Ltd,103,2024-01-25,Office Chair,Furniture,1,350.0,350.0
5,TechStart Ltd,103,2024-01-25,Wireless Mouse,Electronics,5,25.0,125.0
6,Global Industries,104,2024-03-10,Laptop Pro,Electronics,1,1200.0,1200.0
7,Global Industries,104,2024-03-10,"Monitor 27""",Electronics,1,400.0,400.0
8,TechStart Ltd,105,2024-03-15,Office Chair,Furniture,2,350.0,700.0
9,TechStart Ltd,105,2024-03-15,Standing Desk,Furniture,1,600.0,600.0


### 1.4 Aggregations with JOINs

In [28]:
%%sql
-- Customer purchase summary
SELECT 
    c.customer_name,
    c.country,
    COUNT(DISTINCT o.order_id) AS total_orders,
    SUM(oi.quantity * p.price) AS total_spent,
    ROUND(AVG(oi.quantity * p.price), 2) AS avg_order_value
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
LEFT JOIN order_items oi ON o.order_id = oi.order_id
LEFT JOIN products p ON oi.product_id = p.product_id
GROUP BY c.customer_name, c.country
ORDER BY total_spent DESC NULLS LAST;

,customer_name,country,total_orders,total_spent,avg_order_value
0,Acme Corp,USA,2,3275.0,818.75
1,TechStart Ltd,UK,2,1775.0,443.75
2,Global Industries,Japan,1,1600.0,800.00
3,DataCo,USA,0,NaN,NaN
4,Innovate GmbH,Germany,0,NaN,NaN


## 2. Subqueries

A query nested inside another query.

### 2.1 Subquery in WHERE

In [29]:
%%sql
-- Products more expensive than average
SELECT 
    product_name,
    price,
    (SELECT ROUND(AVG(price), 2) FROM products) AS avg_price
FROM products
WHERE price > (SELECT AVG(price) FROM products)
ORDER BY price DESC;

,product_name,price,avg_price
0,Laptop Pro,1200.0,515.0
1,Standing Desk,600.0,515.0


### 2.2 Subquery with IN

In [30]:
%%sql
-- Customers who have placed orders
SELECT customer_name, city
FROM customers
WHERE customer_id IN (SELECT DISTINCT customer_id FROM orders);

,customer_name,city
0,Acme Corp,New York
1,TechStart Ltd,London
2,Global Industries,Tokyo


### 2.3 Correlated Subquery

In [31]:
%%sql
-- For each customer, show their highest order amount
SELECT 
    c.customer_name,
    (
        SELECT MAX(amount) 
        FROM orders o 
        WHERE o.customer_id = c.customer_id
    ) AS highest_order
FROM customers c
WHERE EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id);

,customer_name,highest_order
0,Acme Corp,2300.0
1,TechStart Ltd,1200.0
2,Global Industries,1800.0


## 3. Common Table Expressions (CTE)

CTEs make complex queries more readable by breaking them into named subqueries.

### 3.1 Simple CTE

In [32]:
%%sql
WITH order_totals AS (
    SELECT 
        o.customer_id,
        SUM(oi.quantity * p.price) AS total_spent
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY o.customer_id
)
SELECT 
    c.customer_name,
    c.city,
    ot.total_spent
FROM customers c
JOIN order_totals ot ON c.customer_id = ot.customer_id
ORDER BY ot.total_spent DESC;

,customer_name,city,total_spent
0,Acme Corp,New York,3275.0
1,TechStart Ltd,London,1775.0
2,Global Industries,Tokyo,1600.0


### 3.2 Multiple CTEs

In [33]:
%%sql
WITH customer_stats AS (
    SELECT 
        customer_id,
        COUNT(*) AS order_count,
        SUM(amount) AS total_amount
    FROM orders
    GROUP BY customer_id
),
high_value_customers AS (
    SELECT customer_id
    FROM customer_stats
    WHERE total_amount > 2000
)
SELECT 
    c.customer_name,
    c.country,
    cs.order_count,
    cs.total_amount
FROM customers c
JOIN customer_stats cs ON c.customer_id = cs.customer_id
WHERE c.customer_id IN (SELECT customer_id FROM high_value_customers);

,customer_name,country,order_count,total_amount
0,Acme Corp,USA,2,3800.0
1,TechStart Ltd,UK,2,2150.0


## 4. Window Functions

Window functions perform calculations across rows related to the current row.

### 4.1 ROW_NUMBER

In [34]:
%%sql
-- Rank orders by amount for each customer
SELECT 
    customer_id,
    order_id,
    order_date,
    amount,
    ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY amount DESC) AS rank_in_customer
FROM orders
ORDER BY customer_id, rank_in_customer;

,customer_id,order_id,order_date,amount,rank_in_customer
0,1,102,2024-02-20,2300.0,1
1,1,101,2024-01-15,1500.0,2
2,2,105,2024-03-15,1200.0,1
3,2,103,2024-01-25,950.0,2
4,3,104,2024-03-10,1800.0,1


### 4.2 RANK and DENSE_RANK

In [35]:
%%sql
-- Compare RANK vs DENSE_RANK
SELECT 
    product_name,
    category,
    price,
    ROW_NUMBER() OVER (ORDER BY price DESC) AS row_num,
    RANK() OVER (ORDER BY price DESC) AS rank,
    DENSE_RANK() OVER (ORDER BY price DESC) AS dense_rank
FROM products
ORDER BY price DESC;

,product_name,category,price,row_num,rank,dense_rank
0,Laptop Pro,Electronics,1200.0,1,1,1
1,Standing Desk,Furniture,600.0,2,2,2
2,"Monitor 27""",Electronics,400.0,3,3,3
3,Office Chair,Furniture,350.0,4,4,4
4,Wireless Mouse,Electronics,25.0,5,5,5


### 4.3 Running Totals

In [36]:
%%sql
-- Cumulative order amount over time
SELECT 
    order_date,
    order_id,
    amount,
    SUM(amount) OVER (ORDER BY order_date, order_id) AS running_total
FROM orders
ORDER BY order_date, order_id;

,order_date,order_id,amount,running_total
0,2024-01-15,101,1500.0,1500.0
1,2024-01-25,103,950.0,2450.0
2,2024-02-20,102,2300.0,4750.0
3,2024-03-10,104,1800.0,6550.0
4,2024-03-15,105,1200.0,7750.0


### 4.4 Moving Averages

In [37]:
%%sql
-- 3-order moving average
SELECT 
    order_date,
    order_id,
    amount,
    ROUND(AVG(amount) OVER (
        ORDER BY order_date, order_id 
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 2) AS moving_avg_3
FROM orders
ORDER BY order_date, order_id;

,order_date,order_id,amount,moving_avg_3
0,2024-01-15,101,1500.0,1500.00
1,2024-01-25,103,950.0,1225.00
2,2024-02-20,102,2300.0,1583.33
3,2024-03-10,104,1800.0,1683.33
4,2024-03-15,105,1200.0,1766.67


### 4.5 LAG and LEAD

In [38]:
%%sql
-- Compare each order with the previous one
SELECT 
    order_date,
    order_id,
    amount,
    LAG(amount) OVER (ORDER BY order_date, order_id) AS previous_order_amount,
    amount - LAG(amount) OVER (ORDER BY order_date, order_id) AS difference_from_previous
FROM orders
ORDER BY order_date, order_id;

,order_date,order_id,amount,previous_order_amount,difference_from_previous
0,2024-01-15,101,1500.0,NaN,NaN
1,2024-01-25,103,950.0,1500.0,-550.0
2,2024-02-20,102,2300.0,950.0,1350.0
3,2024-03-10,104,1800.0,2300.0,-500.0
4,2024-03-15,105,1200.0,1800.0,-600.0


## 5. CASE Expressions

CASE provides conditional logic in SQL.

### 5.1 Simple CASE

In [39]:
%%sql
-- Categorize products by price
SELECT 
    product_name,
    price,
    CASE 
        WHEN price >= 1000 THEN 'Premium'
        WHEN price >= 300 THEN 'Mid-range'
        ELSE 'Budget'
    END AS price_category
FROM products
ORDER BY price DESC;

,product_name,price,price_category
0,Laptop Pro,1200.0,Premium
1,Standing Desk,600.0,Mid-range
2,"Monitor 27""",400.0,Mid-range
3,Office Chair,350.0,Mid-range
4,Wireless Mouse,25.0,Budget


### 5.2 CASE in Aggregations

In [40]:
%%sql
-- Conditional counting
SELECT 
    category,
    COUNT(*) AS total_products,
    COUNT(CASE WHEN price >= 500 THEN 1 END) AS expensive_products,
    COUNT(CASE WHEN price < 500 THEN 1 END) AS affordable_products
FROM products
GROUP BY category;

,category,total_products,expensive_products,affordable_products
0,Electronics,3,1,2
1,Furniture,2,1,1


## 6. UNION and UNION ALL

Combine results from multiple queries.

### 6.1 UNION (Removes Duplicates)

In [41]:
%%sql
-- Unique cities from customers and different table
SELECT city, 'Customer' AS source FROM customers
UNION
SELECT 'Paris', 'Other'
ORDER BY city;

,city,source
0,Berlin,Customer
1,London,Customer
2,New York,Customer
3,Paris,Other
4,San Francisco,Customer
5,Tokyo,Customer


### 6.2 UNION ALL (Keeps Duplicates)

In [42]:
%%sql
-- All locations including duplicates
SELECT city, country, 'Customer Location' AS type FROM customers
UNION ALL
SELECT 'London', 'UK', 'Office Location'
ORDER BY city;

,city,country,type
0,Berlin,Germany,Customer Location
1,London,UK,Customer Location
2,London,UK,Office Location
3,New York,USA,Customer Location
4,San Francisco,USA,Customer Location
5,Tokyo,Japan,Customer Location


## 7. Advanced Example: Customer Segmentation

Let's combine multiple concepts:

In [43]:
%%sql
WITH customer_metrics AS (
    SELECT 
        c.customer_id,
        c.customer_name,
        c.country,
        COUNT(DISTINCT o.order_id) AS order_count,
        SUM(oi.quantity * p.price) AS total_spent,
        ROUND(AVG(oi.quantity * p.price), 2) AS avg_order_value
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    LEFT JOIN order_items oi ON o.order_id = oi.order_id
    LEFT JOIN products p ON oi.product_id = p.product_id
    GROUP BY c.customer_id, c.customer_name, c.country
)
SELECT 
    customer_name,
    country,
    COALESCE(order_count, 0) AS orders,
    COALESCE(total_spent, 0) AS revenue,
    COALESCE(avg_order_value, 0) AS avg_value,
    CASE 
        WHEN total_spent >= 3000 THEN 'VIP'
        WHEN total_spent >= 2000 THEN 'Gold'
        WHEN total_spent >= 1000 THEN 'Silver'
        WHEN total_spent > 0 THEN 'Bronze'
        ELSE 'Inactive'
    END AS customer_tier,
    RANK() OVER (ORDER BY COALESCE(total_spent, 0) DESC) AS revenue_rank
FROM customer_metrics
ORDER BY revenue DESC;

,customer_name,country,orders,revenue,avg_value,customer_tier,revenue_rank
0,Acme Corp,USA,2,3275.0,818.75,VIP,1
1,TechStart Ltd,UK,2,1775.0,443.75,Silver,2
2,Global Industries,Japan,1,1600.0,800.00,Silver,3
3,Innovate GmbH,Germany,0,0.0,0.00,Inactive,4
4,DataCo,USA,0,0.0,0.00,Inactive,4


## Clean Up

In [44]:
%%sql
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS order_items;

,Success
